# 10 Recommendation Text Template Refinement

Notebook ini digunakan untuk melakukan finalisasi teks rekomendasi dengan metode **deterministic template mapping**.

Scope perubahan:
- Hanya mengubah kolom `recommendation_text`.
- Tidak mengubah `title`, `category`, `priority_level`, `period_type`, ID, dan kolom relasi.
- Menyimpan `recommendation_text_original` dan `recommendation_template_code` untuk traceability internal.
- Menghasilkan `recommendations_final.csv` sebagai dataset final untuk handoff.


## 1. Import Library dan Setup Path

In [8]:
from pathlib import Path
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    """
    Find project root by walking upward until the expected data/processed folder exists.
    This makes the notebook safe to run from:
    - project root
    - data_analysis/notebooks/01_data_wrangling
    - VS Code/Jupyter working directory variations
    """
    current = start_path.resolve()

    for candidate in [current] + list(current.parents):
        if (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "Project root tidak ditemukan. Pastikan notebook berada di dalam repo "
        "student_stress_data_science dan folder data/processed tersedia."
    )

PROJECT_ROOT = find_project_root(Path.cwd())

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "recommendations_clean.csv"
MAPPING_PATH = PROJECT_ROOT / "data" / "mapping" / "recommendation_text_template_mapping.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "recommendations_final.csv"
REPORT_PATH = PROJECT_ROOT / "outputs" / "reports" / "recommendation_text_refinement_validation.md"

FORBIDDEN_WORDS = ["lo", "gua", "gue", "elo", "kamu", "anda", "pengguna"]

print("Current working directory:", Path.cwd())
print("Detected project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Mapping path:", MAPPING_PATH)
print("Output path:", OUTPUT_PATH)
print("Report path:", REPORT_PATH)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"File input tidak ditemukan: {INPUT_PATH}")

if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"File mapping tidak ditemukan: {MAPPING_PATH}")


Current working directory: c:\Data Codingan\student_stress_data_science\data_analysis\notebooks\01_data_wrangling
Detected project root: C:\Data Codingan\student_stress_data_science
Input path: C:\Data Codingan\student_stress_data_science\data\processed\recommendations_clean.csv
Mapping path: C:\Data Codingan\student_stress_data_science\data\mapping\recommendation_text_template_mapping.csv
Output path: C:\Data Codingan\student_stress_data_science\data\processed\recommendations_final.csv
Report path: C:\Data Codingan\student_stress_data_science\outputs\reports\recommendation_text_refinement_validation.md


## 2. Load Dataset dan Mapping

In [9]:
df = pd.read_csv(INPUT_PATH)
mapping = pd.read_csv(MAPPING_PATH)

print("Dataset shape:", df.shape)
print("Mapping shape:", mapping.shape)

display(df.head())
display(mapping.head())


Dataset shape: (26784, 10)
Mapping shape: (12, 7)


,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at
0,1,1,1.0,NaN,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-02 00:03:00
1,2,1,2.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-02 20:48:00
2,3,1,3.0,NaN,daily,digital_habit,Batasi screen time,Screen time lo tinggi. Kurangi penggunaan laya...,Medium,2026-01-03 20:54:00
3,4,1,4.0,NaN,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-05 01:01:00
4,5,1,5.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-05 23:14:00


,template_code,title,category,priority_level,period_type,original_recommendation_text,revised_recommendation_text
0,RECO_CAFFEINE_MEDIUM_DAILY,Kurangi kafein,caffeine,Medium,daily,Konsumsi kafein lo tinggi. Hindari kafein sore...,Konsumsi kafein tercatat tinggi. Perlu pembata...
1,RECO_DIGITAL_HABIT_MEDIUM_DAILY,Batasi screen time,digital_habit,Medium,daily,Screen time lo tinggi. Kurangi penggunaan laya...,Screen time tercatat tinggi. Perlu pengurangan...
2,RECO_FINANCIAL_HABIT_MEDIUM_DAILY,Atur beban finansial,financial_habit,Medium,daily,Kekhawatiran finansial lo tinggi. Catat pengel...,Kekhawatiran finansial tercatat tinggi. Perlu ...
3,RECO_HEALTH_HIGH_DAILY,Prioritaskan pemulihan,health,High,daily,Kondisi kesehatan lo sedang kurang baik. Kuran...,Kondisi kesehatan sedang kurang baik. Perlu pe...
4,RECO_MAINTENANCE_LOW_DAILY,Pertahankan rutinitas,maintenance,Low,daily,Aktivitas lo relatif stabil. Pertahankan pola ...,Aktivitas harian relatif stabil. Pola tidur da...


## 3. Validasi Kolom Wajib

In [10]:
required_input_columns = {
    "id",
    "period_type",
    "category",
    "title",
    "recommendation_text",
    "priority_level",
}

required_mapping_columns = {
    "template_code",
    "title",
    "category",
    "priority_level",
    "period_type",
    "original_recommendation_text",
    "revised_recommendation_text",
}

missing_input = required_input_columns - set(df.columns)
missing_mapping = required_mapping_columns - set(mapping.columns)

if missing_input:
    raise ValueError(f"Kolom input tidak lengkap: {sorted(missing_input)}")

if missing_mapping:
    raise ValueError(f"Kolom mapping tidak lengkap: {sorted(missing_mapping)}")

print("Validasi kolom wajib aman.")


Validasi kolom wajib aman.


## 4. Cek Unique Recommendation Text

In [11]:
unique_recommendation_text = (
    df[["title", "category", "priority_level", "period_type", "recommendation_text"]]
    .drop_duplicates()
    .sort_values(["category", "priority_level", "title"])
)

print("Jumlah unique recommendation context:", len(unique_recommendation_text))
display(unique_recommendation_text)


Jumlah unique recommendation context: 12


,title,category,priority_level,period_type,recommendation_text
613,Kurangi kafein,caffeine,Medium,daily,Konsumsi kafein lo tinggi. Hindari kafein sore...
2,Batasi screen time,digital_habit,Medium,daily,Screen time lo tinggi. Kurangi penggunaan laya...
196,Atur beban finansial,financial_habit,Medium,daily,Kekhawatiran finansial lo tinggi. Catat pengel...
218,Prioritaskan pemulihan,health,High,daily,Kondisi kesehatan lo sedang kurang baik. Kuran...
5,Pertahankan rutinitas,maintenance,Low,daily,Aktivitas lo relatif stabil. Pertahankan pola ...
0,Stabilkan mood,mood_regulation,Medium,daily,Mood lo sedang rendah. Coba journaling singkat...
87,Aktivitas ringan,physical_activity,Medium,daily,Aktivitas fisik lo rendah. Coba jalan ringan 1...
21,Jadwalkan istirahat,recovery,High,daily,Tingkat kelelahan lo tinggi. Ambil jeda pendek...
2184,Tidur lebih awal,sleep,High,daily,Tidur lo kurang dari kebutuhan ideal. Coba tar...
177,Tidur lebih awal,sleep,Medium,daily,Tidur lo kurang dari kebutuhan ideal. Coba tar...


## 5. Validasi Mapping

Mapping menggunakan key gabungan:

```text
title + category + priority_level + period_type + original_recommendation_text
```

Alasannya: terdapat teks rekomendasi yang sama tetapi memiliki priority berbeda, sehingga mapping tidak boleh hanya berdasarkan `recommendation_text`.


In [12]:
mapping_key_columns = [
    "title",
    "category",
    "priority_level",
    "period_type",
    "original_recommendation_text",
]

duplicated_mapping = mapping[mapping[mapping_key_columns].duplicated(keep=False)]

if not duplicated_mapping.empty:
    display(duplicated_mapping)
    raise ValueError("Mapping memiliki key duplikat.")

print("Tidak ada key mapping duplikat.")


Tidak ada key mapping duplikat.


## 6. Validasi Kata Terlarang pada Template

In [13]:
def validate_no_forbidden_words(series: pd.Series) -> list[str]:
    violations = []
    separators = [",", ".", ";", ":", "!", "?", "(", ")", "[", "]", "{", "}", "/", "\\", "\n", "\t"]

    for value in series.dropna().astype(str).unique():
        lower_value = value.lower()
        for separator in separators:
            lower_value = lower_value.replace(separator, " ")
        tokens = lower_value.split()

        for word in FORBIDDEN_WORDS:
            if word in tokens:
                violations.append(f"{word}: {value}")

    return violations


template_violations = validate_no_forbidden_words(mapping["revised_recommendation_text"])

if template_violations:
    for violation in template_violations:
        print(violation)
    raise ValueError("Template masih mengandung kata terlarang.")

print("Template aman dari kata terlarang.")


Template aman dari kata terlarang.


## 7. Apply Mapping ke Dataset Rekomendasi

In [14]:
key_columns = ["title", "category", "priority_level", "period_type", "recommendation_text"]

mapping_for_merge = mapping.rename(
    columns={"original_recommendation_text": "recommendation_text"}
)

original_columns = df.columns.tolist()

final_df = df.merge(
    mapping_for_merge[
        key_columns + ["template_code", "revised_recommendation_text"]
    ],
    on=key_columns,
    how="left",
    validate="many_to_one",
)

unmapped_df = final_df[final_df["revised_recommendation_text"].isna()]

if not unmapped_df.empty:
    display(
        unmapped_df[
            ["title", "category", "priority_level", "period_type", "recommendation_text"]
        ]
        .drop_duplicates()
        .head(20)
    )
    raise ValueError(f"Ada {len(unmapped_df)} baris recommendation_text yang belum termapping.")

final_df["recommendation_text_original"] = final_df["recommendation_text"]
final_df["recommendation_template_code"] = final_df["template_code"]
final_df["recommendation_text"] = final_df["revised_recommendation_text"]

final_df = final_df.drop(columns=["template_code", "revised_recommendation_text"])
final_df = final_df[original_columns + ["recommendation_text_original", "recommendation_template_code"]]

print("Mapping berhasil diterapkan.")
display(final_df.head())


Mapping berhasil diterapkan.


,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at,recommendation_text_original,recommendation_template_code
0,1,1,1.0,NaN,daily,mood_regulation,Stabilkan mood,Mood tercatat sedang rendah. Perlu aktivitas p...,Medium,2026-01-02 00:03:00,Mood lo sedang rendah. Coba journaling singkat...,RECO_MOOD_REGULATION_MEDIUM_DAILY
1,2,1,2.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik tercatat tinggi. Perlu pembag...,High,2026-01-02 20:48:00,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,RECO_WORKLOAD_HIGH_DAILY
2,3,1,3.0,NaN,daily,digital_habit,Batasi screen time,Screen time tercatat tinggi. Perlu pengurangan...,Medium,2026-01-03 20:54:00,Screen time lo tinggi. Kurangi penggunaan laya...,RECO_DIGITAL_HABIT_MEDIUM_DAILY
3,4,1,4.0,NaN,daily,mood_regulation,Stabilkan mood,Mood tercatat sedang rendah. Perlu aktivitas p...,Medium,2026-01-05 01:01:00,Mood lo sedang rendah. Coba journaling singkat...,RECO_MOOD_REGULATION_MEDIUM_DAILY
4,5,1,5.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik tercatat tinggi. Perlu pembag...,High,2026-01-05 23:14:00,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,RECO_WORKLOAD_HIGH_DAILY


## 8. Validasi Kolom yang Tidak Boleh Berubah

In [15]:
locked_columns = [
    col
    for col in [
        "id",
        "user_id",
        "stress_prediction_id",
        "weekly_summary_id",
        "period_type",
        "category",
        "title",
        "priority_level",
        "created_at",
    ]
    if col in df.columns
]

locked_changed = {}

for col in locked_columns:
    if not df[col].equals(final_df[col]):
        locked_changed[col] = int((df[col] != final_df[col]).sum())

if locked_changed:
    raise ValueError(f"Kolom yang seharusnya tidak berubah ternyata berubah: {locked_changed}")

print("Kolom terkunci tidak berubah:")
print(locked_columns)


Kolom terkunci tidak berubah:
['id', 'user_id', 'stress_prediction_id', 'weekly_summary_id', 'period_type', 'category', 'title', 'priority_level', 'created_at']


## 9. Validasi Output Final

In [16]:
final_violations = validate_no_forbidden_words(final_df["recommendation_text"])

checks = {
    "input_rows": len(df),
    "output_rows": len(final_df),
    "unique_original_recommendation_text": int(df["recommendation_text"].nunique()),
    "mapping_rows": len(mapping),
    "unique_revised_recommendation_text": int(mapping["revised_recommendation_text"].nunique()),
    "missing_revised_text": int(final_df["recommendation_text"].isna().sum()),
    "unmapped_original_text_count": int(unmapped_df.shape[0]),
    "duplicate_id_count": int(final_df["id"].duplicated().sum()) if "id" in final_df else None,
    "forbidden_word_violation_count": len(final_violations),
}

pd.Series(checks)


input_rows                             26784
output_rows                            26784
unique_original_recommendation_text       11
mapping_rows                              12
unique_revised_recommendation_text        12
missing_revised_text                       0
unmapped_original_text_count               0
duplicate_id_count                         0
forbidden_word_violation_count             0
dtype: int64

In [17]:
if checks["input_rows"] != checks["output_rows"]:
    raise ValueError("Jumlah baris input dan output tidak sama.")

if checks["missing_revised_text"] != 0:
    raise ValueError("Masih ada recommendation_text kosong.")

if checks["duplicate_id_count"] != 0:
    raise ValueError("Masih ada duplicate id.")

if checks["forbidden_word_violation_count"] != 0:
    print(final_violations[:20])
    raise ValueError("Output final masih mengandung kata terlarang.")

print("Validasi output final aman.")


Validasi output final aman.


## 10. Distribusi Final

Distribusi ini digunakan untuk memastikan label analitik tetap konsisten setelah refinement.


In [18]:
print("Distribusi period_type:")
display(final_df["period_type"].value_counts(dropna=False))

print("Distribusi category:")
display(final_df["category"].value_counts(dropna=False))

print("Distribusi priority_level:")
display(final_df["priority_level"].value_counts(dropna=False))

print("Distribusi title:")
display(final_df["title"].value_counts(dropna=False))


Distribusi period_type:


period_type
daily     25951
weekly      833
Name: count, dtype: int64

Distribusi category:


category
workload             12020
mood_regulation       5284
maintenance           2825
recovery              1773
sleep                 1259
digital_habit         1132
weekly_target          833
physical_activity      771
financial_habit        533
health                 222
caffeine               132
Name: count, dtype: int64

Distribusi priority_level:


priority_level
High      14870
Medium     9089
Low        2825
Name: count, dtype: int64

Distribusi title:


title
Atur prioritas tugas          12020
Stabilkan mood                 5284
Pertahankan rutinitas          2825
Jadwalkan istirahat            1773
Tidur lebih awal               1259
Batasi screen time             1132
Fokus pemulihan minggu ini      833
Aktivitas ringan                771
Atur beban finansial            533
Prioritaskan pemulihan          222
Kurangi kafein                  132
Name: count, dtype: int64

## 11. Sample Perbandingan Teks Original vs Final

In [19]:
comparison_sample = (
    final_df[
        [
            "title",
            "category",
            "priority_level",
            "period_type",
            "recommendation_text_original",
            "recommendation_text",
            "recommendation_template_code",
        ]
    ]
    .drop_duplicates()
    .sort_values(["category", "priority_level", "title"])
)

display(comparison_sample)


,title,category,priority_level,period_type,recommendation_text_original,recommendation_text,recommendation_template_code
613,Kurangi kafein,caffeine,Medium,daily,Konsumsi kafein lo tinggi. Hindari kafein sore...,Konsumsi kafein tercatat tinggi. Perlu pembata...,RECO_CAFFEINE_MEDIUM_DAILY
2,Batasi screen time,digital_habit,Medium,daily,Screen time lo tinggi. Kurangi penggunaan laya...,Screen time tercatat tinggi. Perlu pengurangan...,RECO_DIGITAL_HABIT_MEDIUM_DAILY
196,Atur beban finansial,financial_habit,Medium,daily,Kekhawatiran finansial lo tinggi. Catat pengel...,Kekhawatiran finansial tercatat tinggi. Perlu ...,RECO_FINANCIAL_HABIT_MEDIUM_DAILY
218,Prioritaskan pemulihan,health,High,daily,Kondisi kesehatan lo sedang kurang baik. Kuran...,Kondisi kesehatan sedang kurang baik. Perlu pe...,RECO_HEALTH_HIGH_DAILY
5,Pertahankan rutinitas,maintenance,Low,daily,Aktivitas lo relatif stabil. Pertahankan pola ...,Aktivitas harian relatif stabil. Pola tidur da...,RECO_MAINTENANCE_LOW_DAILY
0,Stabilkan mood,mood_regulation,Medium,daily,Mood lo sedang rendah. Coba journaling singkat...,Mood tercatat sedang rendah. Perlu aktivitas p...,RECO_MOOD_REGULATION_MEDIUM_DAILY
87,Aktivitas ringan,physical_activity,Medium,daily,Aktivitas fisik lo rendah. Coba jalan ringan 1...,Aktivitas fisik tercatat rendah. Perlu aktivit...,RECO_PHYSICAL_ACTIVITY_MEDIUM_DAILY
21,Jadwalkan istirahat,recovery,High,daily,Tingkat kelelahan lo tinggi. Ambil jeda pendek...,Tingkat kelelahan tercatat tinggi. Perlu jeda ...,RECO_RECOVERY_HIGH_DAILY
2184,Tidur lebih awal,sleep,High,daily,Tidur lo kurang dari kebutuhan ideal. Coba tar...,Durasi tidur berada jauh di bawah kebutuhan id...,RECO_SLEEP_HIGH_DAILY
177,Tidur lebih awal,sleep,Medium,daily,Tidur lo kurang dari kebutuhan ideal. Coba tar...,Durasi tidur berada di bawah kebutuhan ideal. ...,RECO_SLEEP_MEDIUM_DAILY


## 12. Export Dataset Final dan Report Validasi

In [20]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(OUTPUT_PATH, index=False)

report = [
    "# Recommendation Text Refinement Validation Report",
    "",
    "## Tujuan",
    "Dokumen ini mencatat validasi hasil refinement teks rekomendasi berbasis deterministic template mapping. Proses ini hanya mengubah teks display rekomendasi dan tidak mengubah label analitik maupun relasi utama dataset.",
    "",
    "## Input dan Output",
    f"- Input: `{INPUT_PATH}`",
    f"- Mapping: `{MAPPING_PATH}`",
    f"- Output: `{OUTPUT_PATH}`",
    "- Metode: `deterministic_template_mapping_v1`",
    "",
    "## Scope Perubahan",
    "- Kolom `recommendation_text` pada output final berisi teks formal hasil mapping.",
    "- Kolom `title`, `category`, `priority_level`, dan `period_type` tidak diubah agar hasil EDA tetap konsisten.",
    "- Kolom `recommendation_text_original` disimpan untuk traceability internal.",
    "- Kolom `recommendation_template_code` disimpan agar pola template dapat diaudit.",
    "",
    "## Ringkasan Validasi",
    "| Validasi | Nilai |",
    "|---|---:|",
    f"| Jumlah baris input | {checks['input_rows']} |",
    f"| Jumlah baris output | {checks['output_rows']} |",
    f"| Jumlah unique original recommendation_text | {checks['unique_original_recommendation_text']} |",
    f"| Jumlah baris mapping | {checks['mapping_rows']} |",
    f"| Jumlah unique revised recommendation_text | {checks['unique_revised_recommendation_text']} |",
    f"| Missing revised recommendation_text | {checks['missing_revised_text']} |",
    f"| Original recommendation_text yang tidak termapping | {checks['unmapped_original_text_count']} |",
    f"| Duplicate id pada output | {checks['duplicate_id_count']} |",
    f"| Pelanggaran kata terlarang | {checks['forbidden_word_violation_count']} |",
    "",
    "## Distribusi Period Type",
    final_df["period_type"].value_counts(dropna=False).to_markdown(),
    "",
    "## Distribusi Category",
    final_df["category"].value_counts(dropna=False).to_markdown(),
    "",
    "## Distribusi Priority Level",
    final_df["priority_level"].value_counts(dropna=False).to_markdown(),
    "",
    "## Catatan",
    "- Template tidak menggunakan kata subjektif langsung seperti `pengguna`, `kamu`, `Anda`, `lo`, atau `gua`.",
    "- Perubahan ini tidak mengubah struktur rekomendasi, label EDA, maupun relasi ke tabel lain.",
]

REPORT_PATH.write_text("\n".join(report), encoding="utf-8")

print("Dataset final berhasil dibuat:", OUTPUT_PATH)
print("Report validasi berhasil dibuat:", REPORT_PATH)


Dataset final berhasil dibuat: C:\Data Codingan\student_stress_data_science\data\processed\recommendations_final.csv
Report validasi berhasil dibuat: C:\Data Codingan\student_stress_data_science\outputs\reports\recommendation_text_refinement_validation.md
